In [1]:
import pandas as pd
import numpy as np
import time

from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix

In [2]:
caminho_pasta_tratado = '../../dataset tratado/lycos-cicids2017/Sem Redução de Dimensionalidade'

nome_dados_treinamento = 'lycos_cicids2017_treinamento_sem_XSS.csv'
nome_dados_teste = 'lycos_cicids2017_teste_com_XSS.csv'

In [3]:
print("Carregando os datasets tratados em CSV...")
df_treino = pd.read_csv(caminho_pasta_tratado + "/" + nome_dados_treinamento)
df_teste = pd.read_csv(caminho_pasta_tratado + "/" + nome_dados_teste)

print(f"Dados carregados! Treino: {df_treino.shape} | Teste: {df_teste.shape}")
display(df_treino.head())

Carregando os datasets tratados em CSV...
Dados carregados! Treino: (1285780, 80) | Teste: (551718, 80)


,src_port,dst_port,ip_prot,timestamp,flow_duration,down_up_ratio,pkt_len_max,pkt_len_min,pkt_len_mean,pkt_len_var,...,bwd_bulk_bytes_mean,bwd_bulk_pkt_mean,bwd_bulk_rate_mean,fwd_subflow_bytes_mean,fwd_subflow_pkt_mean,bwd_subflow_bytes_mean,bwd_subflow_pkt_mean,fwd_tcp_init_win_bytes,bwd_tcp_init_win_bytes,Label
0,0.951110,0.000809,0.122137,0.493151,0.000504,0.123909,0.003948,0.031768,0.030108,0.000050,...,0.000000,0.000000,0.000000,0.000026,0.000005,1.495151e-07,0.000003,0.000000,0.000000,benign
1,0.766598,0.006760,0.038168,0.235429,0.025492,0.041303,0.001249,0.000000,0.003345,0.000009,...,0.000000,0.000000,0.000000,0.000009,0.000007,0.000000e+00,0.000002,0.003922,0.021423,benign
2,0.814466,0.006760,0.038168,0.512447,0.979619,0.111518,0.057131,0.000000,0.066731,0.004397,...,0.000004,0.000069,0.000009,0.000228,0.000030,2.463439e-06,0.000021,0.445572,0.647110,benign
3,0.074983,0.000809,0.122137,0.240826,0.000002,0.123909,0.010475,0.022790,0.061262,0.000639,...,0.000000,0.000000,0.000000,0.000037,0.000009,7.933453e-07,0.000007,0.000000,0.000000,benign
4,0.847898,0.006760,0.038168,0.235855,0.070290,0.149890,0.175020,0.000000,0.392759,0.041693,...,0.000387,0.000566,0.000418,0.000322,0.000094,6.456254e-05,0.000086,0.445572,0.176773,benign


In [4]:
X_treino = df_treino.drop('Label', axis=1)
y_treino = df_treino['Label']

X_teste = df_teste.drop('Label', axis=1)
y_teste = df_teste['Label']

print(f"Dados prontos! Treino: {X_treino.shape} | Teste: {X_teste.shape}")

Dados prontos! Treino: (1285780, 79) | Teste: (551718, 79)


In [5]:
rf_model = RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1)

print("\nIniciando o treinamento do Random Forest...")
inicio_treino = time.time()

rf_model.fit(X_treino, y_treino)

fim_treino = time.time()
tempo_treinamento = fim_treino - inicio_treino
print(f"Treinamento concluído! Tempo total: {tempo_treinamento:.4f} segundos.")


Iniciando o treinamento do Random Forest...
Treinamento concluído! Tempo total: 71.6156 segundos.


In [6]:
print("\nIniciando a classificação dos dados de teste...")
inicio_teste = time.time()

previsoes = rf_model.predict(X_teste)

fim_teste = time.time()
tempo_teste = fim_teste - inicio_teste
print(f"Classificação concluída! Tempo de previsão: {tempo_teste:.4f} segundos.")

print("\n=== MATRIZ DE CONFUSÃO (Sem XSS no treino) ===")
labels = sorted(np.unique(np.concatenate([y_teste.values, previsoes])))
cm = confusion_matrix(y_teste, previsoes, labels=labels)

cm_df = pd.DataFrame(
    cm,
    index=[f"Real_{lbl}" for lbl in labels],
    columns=[f"Pred_{lbl}" for lbl in labels]
)

def color_confusion_matrix(data):
    styles = pd.DataFrame("", index=data.index, columns=data.columns)
    for i in range(min(data.shape[0], data.shape[1])):
        styles.iat[i, i] = "background-color: #c6efce; color: #006100; font-weight: bold;"
    for i in range(data.shape[0]):
        for j in range(data.shape[1]):
            if i != j and data.iat[i, j] != 0:
                styles.iat[i, j] = "background-color: #ffc7ce; color: #9c0006; font-weight: bold;"
    return styles

display(cm_df.style.apply(color_confusion_matrix, axis=None).format("{:.0f}"))

print("\n=== RELATÓRIO DE MÉTRICAS (Precisão, Recall, F1-Score) ===")
print(classification_report(y_teste, previsoes, zero_division=0))


Iniciando a classificação dos dados de teste...
Classificação concluída! Tempo de previsão: 1.9077 segundos.

=== MATRIZ DE CONFUSÃO (Sem XSS no treino) ===


,Pred_benign,Pred_bot,Pred_ddos,Pred_dos_goldeneye,Pred_dos_hulk,Pred_dos_slowhttptest,Pred_dos_slowloris,Pred_ftp_patator,Pred_heartbleed,Pred_portscan,Pred_ssh_patator,Pred_webattack_bruteforce,Pred_webattack_sql_injection,Pred_webattack_xss
Real_benign,418625,0,0,9,1,3,0,0,0,0,0,1,0,0
Real_bot,0,219,0,0,0,0,0,0,0,0,0,0,0,0
Real_ddos,0,0,28596,0,0,0,0,0,0,0,0,0,0,0
Real_dos_goldeneye,1,0,0,2095,0,0,0,0,0,0,0,0,0,0
Real_dos_hulk,2,0,0,0,47940,0,0,0,0,0,0,0,0,0
Real_dos_slowhttptest,3,0,0,1,0,1419,0,0,0,0,0,0,0,0
Real_dos_slowloris,3,0,0,0,0,2,1769,0,0,0,0,1,0,0
Real_ftp_patator,0,0,0,0,0,0,0,1181,0,0,0,0,0,0
Real_heartbleed,1,0,0,0,0,0,0,0,3,0,0,0,0,0
Real_portscan,1,0,0,0,0,0,0,0,0,47906,0,0,0,0



=== RELATÓRIO DE MÉTRICAS (Precisão, Recall, F1-Score) ===
                         precision    recall  f1-score   support

                 benign       1.00      1.00      1.00    418639
                    bot       1.00      1.00      1.00       219
                   ddos       1.00      1.00      1.00     28596
          dos_goldeneye       1.00      1.00      1.00      2096
               dos_hulk       1.00      1.00      1.00     47942
       dos_slowhttptest       1.00      1.00      1.00      1423
          dos_slowloris       1.00      1.00      1.00      1775
            ftp_patator       1.00      1.00      1.00      1181
             heartbleed       1.00      0.75      0.86         4
               portscan       1.00      1.00      1.00     47907
            ssh_patator       1.00      1.00      1.00       875
   webattack_bruteforce       0.68      0.98      0.80       397
webattack_sql_injection       1.00      0.67      0.80         3
          webattack_xss      

In [7]:
CLASS_ALIASES_LATEX = {'benign': 'BENIGN', 'bot': 'Bot', 'ddos': 'DDoS', 'dos_goldeneye': 'DoS GE', 'dos_hulk': 'Hulk', 'dos_slowhttptest': 'Slowhttp', 'dos_slowloris': 'Slowloris', 'ftp_patator': 'FTP', 'heartbleed': 'Heart', 'portscan': 'PortScan', 'ssh_patator': 'SSH', 'webattack_bruteforce': 'Brute', 'webattack_sql_injection': 'SQL', 'webattack_xss': 'XSS'}


def escape_latex(value):
    replacements = {
        "\\": "\\textbackslash{}",
        "&": "\\&",
        "%": "\\%",
        "$": "\\$",
        "#": "\\#",
        "_": "\\_",
        "{": "\\{",
        "}": "\\}",
        "~": "\\textasciitilde{}",
        "^": "\\textasciicircum{}",
    }
    return "".join(replacements.get(char, char) for char in str(value))


def latex_class_name(label, short=False):
    if short:
        return escape_latex(CLASS_ALIASES_LATEX.get(label, label))
    return escape_latex(label)


def format_confusion_value(value, is_diagonal):
    value = int(value)
    if is_diagonal:
        return f"\\ok{{{value}}}"
    if value != 0:
        return f"\\err{{{value}}}"
    return "0"


def make_latex_confusion_matrix(cm_values, class_labels, caption, table_label):
    headers = [latex_class_name(label, short=True) for label in class_labels]
    rows = []
    for i, real_label in enumerate(class_labels):
        row_values = [format_confusion_value(cm_values[i][j], i == j) for j in range(len(class_labels))]
        rows.append((f"Real\\_{latex_class_name(real_label, short=True)}", row_values))

    first_col_width = max([0] + [len(row_name) for row_name, _ in rows])
    col_widths = [max(len(headers[i]), *(len(values[i]) for _, values in rows)) for i in range(len(headers))]

    def format_row(first_cell, values):
        first = first_cell.ljust(first_col_width)
        rest = " & ".join(str(value).ljust(col_widths[i]) for i, value in enumerate(values))
        return f"            {first} & {rest} \\\\"

    lines = [
        "\\begin{table}[H]",
        "    \\centering",
        "    \\scriptsize",
        "    \\resizebox{\\linewidth}{!}{",
        f"        \\begin{{tabular}}{{l|{'r' * len(class_labels)}}}",
        "            \\hline",
        format_row("", headers),
        "            \\hline",
    ]
    lines.extend(format_row(row_name, row_values) for row_name, row_values in rows)
    lines.extend([
        "            \\hline",
        "        \\end{tabular}",
        "    }",
        f"    \\caption{{{escape_latex(caption)}}}",
        f"    \\label{{{table_label}}}",
        "\\end{table}",
    ])
    return "\n".join(lines)


def format_metric(value):
    return "-" if value is None else f"{value:.2f}"


def make_latex_metrics_report(y_true_values, y_pred_values, class_labels, caption, table_label):
    report = classification_report(
        y_true_values,
        y_pred_values,
        labels=class_labels,
        output_dict=True,
        zero_division=0,
    )
    total_support = int(sum(report[label]["support"] for label in class_labels))
    rows = []
    for label in class_labels:
        metrics = report[label]
        rows.append([
            latex_class_name(label),
            format_metric(metrics["precision"]),
            format_metric(metrics["recall"]),
            format_metric(metrics["f1-score"]),
            str(int(metrics["support"])),
        ])

    rows.extend([
        ["\\textbf{Acurácia}", "-", format_metric(report["accuracy"]), "-", str(total_support)],
        ["\\textbf{Média Macro}", format_metric(report["macro avg"]["precision"]), format_metric(report["macro avg"]["recall"]), format_metric(report["macro avg"]["f1-score"]), str(total_support)],
        ["\\textbf{Média Ponderada}", format_metric(report["weighted avg"]["precision"]), format_metric(report["weighted avg"]["recall"]), format_metric(report["weighted avg"]["f1-score"]), str(total_support)],
    ])

    headers = ["Classe", "Precisão", "Revocação", "F1-score", "Suporte"]
    col_widths = [max(len(str(row[i])) for row in [headers] + rows) for i in range(len(headers))]

    def format_row(values):
        return "        " + " & ".join(str(value).ljust(col_widths[i]) for i, value in enumerate(values)) + " \\\\"

    lines = [
        "\\begin{table}[H]",
        "    \\centering",
        "    \\small",
        "    \\begin{tabular}{lrrrr}",
        "        \\hline",
        format_row(headers),
        "        \\hline",
    ]
    lines.extend(format_row(row) for row in rows[:len(class_labels)])
    lines.extend([
        "        \\hline",
        format_row(rows[-3]),
        format_row(rows[-2]),
        format_row(rows[-1]),
        "        \\hline",
        "    \\end{tabular}",
        f"    \\caption{{{escape_latex(caption)}}}",
        f"    \\label{{{table_label}}}",
        "\\end{table}",
    ])
    return "\n".join(lines)


labels_latex = labels
y_true_latex = y_teste
y_pred_latex = previsoes

latex_mc = make_latex_confusion_matrix(
    cm, labels_latex,
    "Matriz de Confusão — Random Forest (LycoS-IDS2017, Sem XSS no treino)",
    "table:rf_lycos_sem_xss_mc",
)
latex_metricas = make_latex_metrics_report(
    y_true_latex, y_pred_latex, labels_latex,
    "Relatório de Métricas — Random Forest (LycoS-IDS2017, Sem XSS no treino)",
    "table:rf_lycos_sem_xss_metricas",
)

print(latex_mc)
print("\n")
print(latex_metricas)

\begin{table}[H]
    \centering
    \scriptsize
    \resizebox{\linewidth}{!}{
        \begin{tabular}{l|rrrrrrrrrrrrrr}
            \hline
                            & BENIGN      & Bot      & DDoS       & DoS GE    & Hulk       & Slowhttp  & Slowloris & FTP       & Heart  & PortScan   & SSH      & Brute     & SQL    & XSS    \\
            \hline
            Real\_BENIGN    & \ok{418625} & 0        & 0          & \err{9}   & \err{1}    & \err{3}   & 0         & 0         & 0      & 0          & 0        & \err{1}   & 0      & 0      \\
            Real\_Bot       & 0           & \ok{219} & 0          & 0         & 0          & 0         & 0         & 0         & 0      & 0          & 0        & 0         & 0      & 0      \\
            Real\_DDoS      & 0           & 0        & \ok{28596} & 0         & 0          & 0         & 0         & 0         & 0      & 0          & 0        & 0         & 0      & 0      \\
            Real\_DoS GE    & \err{1}     & 0        & 0          & \